In [ ]:
import os
import random
import scipy.io
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
from FreeHybridKANSRF import FreeHybridColorKAN



In [ ]:
number_of_primitives = 3
visible_range = True#Use False to select entire spectrum
include_shadows = True
no_of_epochs = 30
#Visible range of pavia dataset has 66 bands and full spectrum has 103 bands
no_of_bands = 66 if visible_range else 103
output_classes = 9 if include_shadows else 8

In [ ]:
hsi = scipy.io.loadmat("PaviaU.mat")
image = hsi["paviaU"]

gt = scipy.io.loadmat("PaviaU_gt.mat")
index = gt["paviaU_gt"]
image  =image[:, :, :no_of_bands]

print(image.shape)
print(index.shape)

In [ ]:
#Normalizing the image
max_val = np.max(image)
normalized_image = image.astype(np.float32) / max_val

In [ ]:

image_for_training = normalized_image[:,:,:no_of_bands]
print(image_for_training.shape)

In [ ]:
x = []
for i in range(len(image_for_training)):
    for j in range(len(image_for_training[0])):
        if (index[i][j] == 0) or (index[i][j] == 9 and (not include_shadows)): #Ignoring all unclassified pixels
            continue
        x.append(image_for_training[i][j])

x = np.asarray(x,dtype=np.float32)
print(x.shape)

In [ ]:
y = []

for i in range(len(image)):
    for j in range(len(image[0])):
        if index[i][j] == 0 or \
           (index[i][j] == 9 and (not include_shadows)):
            continue

        y.append(index[i][j]-1)

y = np.asarray(y , dtype=np.int64)
y.shape

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)
model = FreeHybridColorKAN(
    no_of_bands=no_of_bands,
    number_of_primitives=number_of_primitives,
    output_classes=output_classes,

    # Keep the original FastKAN intensity capacity
    intensity_num_grids=8,

    use_base_update=True,
    spline_weight_init_scale=0.1,

    # Initialization of the 66 independent envelope weights
    envelope_init_scale=0.1
).to(device)

print(model)

In [ ]:
from torchinfo import summary
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = model.to(device)

summary(
    model,
    input_size=(1, no_of_bands))

optimizer = torch.optim.Adam(
    model.parameters(),)

criterion = nn.CrossEntropyLoss()

x_tensor = torch.tensor( x,dtype=torch.float32)

y_tensor = torch.tensor( y,dtype=torch.long)

batch_size = 32

dataset = TensorDataset(x_tensor,y_tensor)

train_loader = DataLoader(dataset,batch_size=batch_size,shuffle=True)

loss_history = []
accuracy_history = []

for epoch in range(no_of_epochs):

    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()

        output = model(batch_x)

        loss = criterion(output,batch_y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

        predictions = torch.argmax(output,dim=1)

        total_correct += ( predictions == batch_y).sum().item()

        total_samples += batch_y.size(0)

    epoch_loss = total_loss / total_samples

    epoch_accuracy = (total_correct / total_samples
    )

    loss_history.append(
        epoch_loss
    )

    accuracy_history.append(
        epoch_accuracy
    )

    print(
        f"Epoch [{epoch+1}/{no_of_epochs}] | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.4f}"
    )

In [ ]:
plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.plot(loss_history)
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1, 2, 2)
plt.plot(accuracy_history)
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.tight_layout()
plt.show()

In [ ]:
wavelength = np.linspace(430, 860, no_of_bands) #430 - 860 is range of paviaU dataset and 103 is number of bands
wavelength_int = wavelength.astype(int)
print(wavelength_int)

In [ ]:
primitive_curves = []

model.eval()

for primitive_no in range(number_of_primitives):

    with torch.no_grad():

        curve = (
            model.primitives[primitive_no]
            .get_srf()
            .cpu()
            .numpy()
        )

    primitive_curves.append(curve)

    print(
        f"Primitive {primitive_no + 1} "
        f"SRF shape: {curve.shape}"
    )

    plt.figure(figsize=(10, 5))

    plt.plot(
        wavelength_int[:no_of_bands],
        curve,
        linewidth=2
    )

    plt.title(
        f"Hybrid-KAN-Generated SRF "
        f"Primitive {primitive_no + 1}"
    )

    plt.xlabel("Wavelength (nm)")
    plt.ylabel("SRF Weight")
    plt.tight_layout()
    plt.show()

In [ ]:
for i ,p in enumerate(primitive_curves):
    print(i+1, f"primitive_{i+1}" , p[0])



In [ ]:
for i, (name, parameter) in enumerate(model.named_parameters()):
    if parameter.requires_grad:
        print(i, name, parameter.shape)

In [ ]:

plt.figure(figsize=(10, 5))

for primitive_no in range(number_of_primitives):

    plt.plot(wavelength_int[:no_of_bands],primitive_curves[primitive_no],label=f"Primitive {primitive_no + 1}")
plt.legend(bbox_to_anchor=(1.25, 1))
plt.show()




In [ ]:
model_name = (
    f'{no_of_bands}bands_'f'{number_of_primitives}primitives_' f'{"with_shadow" if include_shadows else "without_shadow"}_' f'free_envelope_hybrid_')
print(model_name)

In [ ]:
print(model_name)

In [ ]:
torch.save(model.state_dict(), f"{model_name}.pth")

In [ ]:
model = FreeHybridColorKAN(
    no_of_bands=no_of_bands,
    number_of_primitives=number_of_primitives,
    output_classes=output_classes,
    intensity_num_grids=8,
    use_base_update=True,
    spline_weight_init_scale=0.1,
    envelope_init_scale=0.1
).to(device)

model.load_state_dict(
    torch.load(
        f"{model_name}.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

In [ ]:
def predict_image():
    x_pred = image_for_training.reshape(-1, no_of_bands)

    print("Prediction Input Shape:", x_pred.shape)
    print("Predicting...")

    x_pred = torch.tensor( x_pred,dtype=torch.float32, device=device)

    model.eval()

    with torch.no_grad():

        primitive_outputs = [primtive(x_pred) for primtive in model.primitives]
        prediction = torch.cat(primitive_outputs,dim=1)

    predicted_image = prediction.cpu().numpy().reshape(
        image.shape[0],
        image.shape[1],
        number_of_primitives
    )

    return predicted_image
def normalize_images(predicted_image, channels):

    max_val = np.max(predicted_image)

    if max_val == 0:
        global_max_image = predicted_image.copy()
    else:
        global_max_image = predicted_image / max_val

    signed_channelwise_image = np.zeros_like(predicted_image , dtype=np.float32)

    for i in range(predicted_image.shape[-1]):

        channel = predicted_image[:, :, i]

        channel_max = np.max(np.abs(channel))

        if channel_max!= 0:
            signed_channelwise_image[:, :, i] = (
                channel / channel_max
            )
        else:
            signed_channelwise_image[:, :, i] = channel
    clipped_image = np.clip(
        predicted_image,0,1)
    channel_minmax_image = np.zeros_like(predicted_image , dtype=np.float32)
    for i in range(predicted_image.shape[-1]):
        channel = predicted_image[: , : , i]
        channel_min = np.min(channel)
        channel_max = np.max(channel)
        channel_minmax_image[: ,: ,i] = (channel -channel_min)/(channel_max -channel_min + 1e-8)
    c1 , c2 , c3 = [channel -1 for channel in channels]

    rgb_global_max = np.dstack((
        global_max_image[:, :, c1],
        global_max_image[:, :, c2],
        global_max_image[:, :, c3]
    ))

    rgb_signed_channelwise = np.dstack((
        signed_channelwise_image[:, :, c1],
        signed_channelwise_image[:, :, c2],
        signed_channelwise_image[:, :, c3]
    ))

    rgb_clipped = np.dstack((
        clipped_image[:, :, c1],
        clipped_image[:, :, c2],
        clipped_image[:, :, c3]
    ))
   
    return [
        rgb_global_max ,
        rgb_signed_channelwise,
        rgb_clipped
    ]

In [ ]:
channels = [1, 2, 3]

predicted_image = predict_image()

normalized_images = normalize_images(
    predicted_image,
    channels
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(normalized_images[0])
plt.title("Global-Max Norm")

plt.subplot(1, 3, 2)
plt.imshow(normalized_images[1])
plt.title("Signed Channel-wise Norm")

plt.subplot(1, 3, 3)
plt.imshow(normalized_images[2])
plt.title("Clipping")

plt.show()